# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

### 1. Target distribution

In [3]:
print(fraud["fraud"].value_counts())
print()
print(fraud["fraud"].value_counts(normalize=True))

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

fraud
0.0    0.912597
1.0    0.087403
Name: proportion, dtype: float64


**Yes, heavily imbalanced**: only **8.7%** of the 1,000,000 transactions are fraudulent (87,403 vs. 912,597 legit). A model that just predicted "never fraud" would already score 91.3% accuracy while catching zero fraud — accuracy alone would be a useless metric here.

### 2. Train a Logistic Regression (on the imbalanced data, as a baseline)

In [4]:
from sklearn.linear_model import LogisticRegression

X = fraud.drop(columns=["fraud"])
y = fraud["fraud"]

# stratify=y keeps the same ~8.7% fraud rate in both train and test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0, stratify=y)

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

### 3. Evaluate (with class imbalance in mind)

In [5]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

def evaluate(name, model, X_te, y_te):
    pred = model.predict(X_te)
    print(f"--- {name} ---")
    print(classification_report(y_te, pred, target_names=["legit", "fraud"]))
    return {
        "precision": precision_score(y_te, pred),
        "recall": recall_score(y_te, pred),
        "f1": f1_score(y_te, pred),
    }

baseline_scores = evaluate("Baseline (imbalanced)", baseline_model, X_test, y_test)

--- Baseline (imbalanced) ---
              precision    recall  f1-score   support

       legit       0.96      0.99      0.98    182519
       fraud       0.90      0.60      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000



**Accuracy is the wrong metric here** — with 91% of transactions legit, a model can score high accuracy while missing most fraud. What matters is **recall on the fraud class** (of all real fraud, how much did we catch — the cost of a missed fraud is real money lost) balanced against **precision** (of all transactions flagged as fraud, how many actually were — the cost of too many false alarms is wasted investigation effort). **F1** is the balance of both in one number.

The baseline: precision 89.5%, but **recall only 60.2%** — this model, trained on data where fraud is rare, learned to be conservative about calling something fraud, and as a result misses **40% of actual fraud cases**. That's the imbalance problem showing up directly: the model barely saw enough fraud examples to learn what they look like.

### 4. Oversample the minority class

In [6]:
from sklearn.utils import resample

train = X_train.copy()
train["fraud"] = y_train.values

majority = train[train["fraud"] == 0]
minority = train[train["fraud"] == 1]

# Oversampling: resample the minority class WITH replacement, up to the
# majority class's size -- duplicates minority rows, doesn't invent new data.
minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=0)
oversampled = pd.concat([majority, minority_upsampled])

X_train_over = oversampled.drop(columns=["fraud"])
y_train_over = oversampled["fraud"]

print(y_train_over.value_counts())

model_over = LogisticRegression(max_iter=1000)
model_over.fit(X_train_over, y_train_over)
over_scores = evaluate("Oversampled", model_over, X_test, y_test)

fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


--- Oversampled ---


              precision    recall  f1-score   support

       legit       0.99      0.93      0.96    182519
       fraud       0.57      0.95      0.71     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



**Recall jumps to 95.0%** (from 60.2%) — the model now sees fraud as often as legit transactions during training, so it's far more willing to flag something as fraud. The tradeoff: **precision drops to 57.2%** (from 89.5%) — nearly half of what it flags as fraud is actually legit. F1 (71.4%) ends up almost identical to the baseline's (72.0%) — oversampling didn't make the model strictly "better," it moved the precision/recall tradeoff hard toward recall, which is usually the right call for fraud detection (a missed fraud costs more than a false alarm), but it is a tradeoff, not a free win.

### 5. Undersample the majority class

In [7]:
# Undersampling: the opposite move -- randomly drop rows from the majority
# class down to the minority class's size, instead of duplicating the minority.
majority_downsampled = resample(majority, replace=False, n_samples=len(minority), random_state=0)
undersampled = pd.concat([majority_downsampled, minority])

X_train_under = undersampled.drop(columns=["fraud"])
y_train_under = undersampled["fraud"]

print(y_train_under.value_counts())

model_under = LogisticRegression(max_iter=1000)
model_under.fit(X_train_under, y_train_under)
under_scores = evaluate("Undersampled", model_under, X_test, y_test)

fraud
0.0    69922
1.0    69922
Name: count, dtype: int64


--- Undersampled ---
              precision    recall  f1-score   support

       legit       0.99      0.93      0.96    182519
       fraud       0.57      0.95      0.71     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



**Nearly identical to oversampling** (precision 57.1%, recall 95.0%, F1 71.3%) — makes sense, since both techniques do the same fundamental thing (rebalance the class ratio the model trains on), just from opposite directions. The practical difference is elsewhere: undersampling threw away roughly 660,000 legit-transaction rows to get there, training on ~140K rows total instead of oversampling's ~1.46M — much faster to train, at the cost of discarding real data that could have helped the model learn.

### 6. SMOTE

In [8]:
from imblearn.over_sampling import SMOTE

# SMOTE doesn't just duplicate minority rows like plain oversampling --
# it generates new synthetic minority samples by interpolating between
# real minority points and their nearest minority neighbors.
smote = SMOTE(random_state=0)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(pd.Series(y_train_smote).value_counts())

model_smote = LogisticRegression(max_iter=1000)
model_smote.fit(X_train_smote, y_train_smote)
smote_scores = evaluate("SMOTE", model_smote, X_test, y_test)

fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


--- SMOTE ---
              precision    recall  f1-score   support

       legit       0.99      0.93      0.96    182519
       fraud       0.57      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



**Essentially the same result again** (precision 57.4%, recall 94.8%, F1 71.5%) — within noise of both plain oversampling and undersampling. SMOTE's synthetic-interpolation approach is often recommended over naive duplication specifically to avoid a model just memorizing repeated identical minority rows, but for a *linear* model like Logistic Regression on this dataset, that distinction barely shows up in the final metrics — the three rebalancing strategies land in nearly the same place.

### Summary

In [9]:
summary = pd.DataFrame(
    {
        "Baseline (imbalanced)": baseline_scores,
        "Oversampled": over_scores,
        "Undersampled": under_scores,
        "SMOTE": smote_scores,
    }
).T.round(4)
summary

,precision,recall,f1
Baseline (imbalanced),0.8950,0.6018,0.7197
Oversampled,0.5723,0.9502,0.7143
Undersampled,0.5713,0.9501,0.7135
SMOTE,0.5740,0.9480,0.7151


**Does balancing improve performance?** Not in the "higher F1" sense — all three balanced approaches land around F1 ≈ 0.71-0.71, statistically indistinguishable from the imbalanced baseline's F1 ≈ 0.72. What balancing *does* change dramatically is **where the model sits on the precision/recall tradeoff**: from a conservative model that catches 60% of fraud with few false alarms, to an aggressive one that catches 95% of fraud at the cost of a lot more false alarms. **Which one is "better" isn't a modeling question, it's a business question** — depends entirely on whether the cost of a missed fraud outweighs the cost of a false alarm for this specific business. For most fraud-detection use cases the answer is yes (a missed fraud is expensive, flagging a legit transaction for review is cheap by comparison), which is why balancing techniques are standard practice here even though they don't move the single-number F1 metric much.